<a href="https://colab.research.google.com/github/nancymatijas/alumni-llm-workshop/blob/main/01_llm_api_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM API Fundamentals: Build an AI Quiz Game

## Overview

In this notebook, we'll learn the fundamentals of interacting with Large Language Models through APIs by building an AI-powered quiz game, step by step.

We'll explore:
- Making API calls and inspecting responses
- Understanding that models are **stateless** — the full conversation must be sent every time
- How system prompts control model behavior
- Getting **structured output** (XML) and parsing it in Python
- **Thinking models** that show their reasoning (and burn tokens doing it)
- Building a complete **Study Buddy** quiz game from your own notes

**What You'll Build:** An AI-powered quiz game that generates questions from YOUR study materials.

**API:** We use [OpenRouter](https://openrouter.ai/), which gives access to many LLM providers through a single OpenAI-compatible API.

**Time:** ~2 hours (6 exercises)

## Setup and Installation

First, let's install the required packages and set up our environment.

In [ ]:
# Install required packages
!pip install openai -q

In [ ]:
import os
import time
import xml.etree.ElementTree as ET
from openai import OpenAI
from google.colab import userdata

# Configure OpenRouter API
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', None)

if not OPENROUTER_API_KEY:
    raise ValueError("Please set your OPENROUTER_API_KEY in Colab secrets (key icon in sidebar).")

# Initialize OpenAI client with OpenRouter endpoint
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Models we'll use throughout the notebook
MODEL_FAST = "stepfun/step-3.5-flash:free"                  # Fast and free
MODEL_MISTRAL = "mistralai/mistral-small-2603"               # Balanced
MODEL_NEMOTRON = "nvidia/nemotron-3-super-120b-a12b:free"    # Large open-weight model
MODEL_THINKING = "openai/gpt-oss-120b"                       # Reasoning/thinking model
MODEL_MINISTRAL = "mistralai/ministral-14b-2512"             # Mid-size Mistral

print("\u2713 OpenRouter client configured")
print(f"\u2713 Models available:")
print(f"  \u2022 {MODEL_FAST} (fast, free)")
print(f"  \u2022 {MODEL_MISTRAL} (balanced)")
print(f"  \u2022 {MODEL_NEMOTRON} (large open-weight)")
print(f"  \u2022 {MODEL_THINKING} (reasoning/thinking)")
print(f"  \u2022 {MODEL_MINISTRAL} (mid-size Mistral)")

✓ OpenRouter client configured
✓ Models available:
  • stepfun/step-3.5-flash:free (fast, free)
  • mistralai/mistral-small-2603 (balanced)
  • nvidia/nemotron-3-super-120b-a12b:free (large open-weight)
  • openai/gpt-oss-120b (reasoning/thinking)
  • mistralai/ministral-14b-2512 (mid-size Mistral)


In [ ]:
def chat(messages, model=MODEL_FAST, temperature=0.7, **kwargs):
    """
    Send messages to an LLM and return the full response object.

    Args:
        messages: List of message dicts with 'role' and 'content'
        model: Model identifier string
        temperature: Sampling temperature (0.0 = deterministic, 1.0+ = creative)
        **kwargs: Additional parameters passed to the API

    Returns:
        The full API response object
    """
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        **kwargs
    )
    return response

print("✓ chat() helper function defined")

✓ chat() helper function defined


---

## Exercise 1: Hello, API

### Goal
Make your first LLM API call, understand the request/response structure, and compare how different models respond to the same prompt.

### Key Concepts
- The Chat Completions API uses a `messages` array
- Each message has a `role` (`"user"`, `"assistant"`, or `"system"`) and `content`
- The response contains the generated text, model info, and **token usage**

In [ ]:
# Your first API call!
messages = [
    {"role": "user", "content": "What is an API? Explain in 2 sentences."}
]

response = chat(messages)

# Let's inspect the FULL response object — this isn't magic, it's a data structure
print("=== Full Response Object ===")
print(f"ID:      {response.id}")
print(f"Model:   {response.model}")
print(f"Created: {response.created}")
print()

# The actual generated text
print("=== Generated Text ===")
print(response.choices[0].message.content)
print()

# Token usage — this is how you get billed!
print("=== Token Usage ===")
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens:      {response.usage.total_tokens}")
print()

import json
print("=== Full JSON Response ===")
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

=== Full Response Object ===
ID:      gen-1774372451-YRn2dGdnQDHPmwivRifc
Model:   stepfun/step-3.5-flash:free
Created: 1774372451

=== Generated Text ===
An API (Application Programming Interface) is a set of rules and tools that allows different software applications to communicate and share data with each other. It acts as a messenger, enabling one program to request services or information from another without needing to understand its internal code.

=== Token Usage ===
Prompt tokens:     23
Completion tokens: 201
Total tokens:      224
=== Full JSON Response ===
{
  "id": "gen-1774372451-YRn2dGdnQDHPmwivRifc",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "An API (Application Programming Interface) is a set of rules and tools that allows different software applications to communicate and share data with each other. It acts as a messenger, enabling one program to request services or information f

### Understanding Tokens

Tokens are the "units" that LLMs process. They're not exactly words — roughly:
- **English:** 100 tokens ≈ 75 words
- **Croatian:** 100 tokens ≈ 40 words (non-English languages use more tokens!)
- `"Hello world"` = 2 tokens
- `"Antiestablishmentarianism"` = 6 tokens

**Why tokens matter:**
1. You **pay per token** (input + output)
2. Every model has a maximum **context window** (total tokens for input + output)
3. Different models have different tokenizers → different token counts for the same text

In [ ]:
# Same prompt, different models — compare responses, tokens, and speed
prompt = "Explain recursion to a 10-year-old in 3 sentences."

models = [MODEL_FAST, MODEL_MISTRAL, MODEL_NEMOTRON, MODEL_MINISTRAL]

for model_name in models:
    print(f"\n{'=' * 70}")
    print(f"Model: {model_name}")
    print('=' * 70)

    start_time = time.time()
    response = chat(
        [{"role": "user", "content": prompt}],
        model=model_name
    )
    elapsed = time.time() - start_time

    print(f"\nResponse:")
    print(response.choices[0].message.content)
    print(f"\nTokens: {response.usage.prompt_tokens} in + {response.usage.completion_tokens} out = {response.usage.total_tokens} total")
    print(f"Time: {elapsed:.2f}s")


Model: stepfun/step-3.5-flash:free

Response:
Imagine a set of Russian nesting dolls—each one opens to reveal a smaller doll inside, until you get to the tiniest one that can’t open anymore.  
Recursion is like that: you solve a big problem by breaking it into a smaller copy of the same problem, and keep going until it’s small enough to solve easily.  
It’s a way of repeating yourself smartly, so you don’t have to figure everything out all at once.

Tokens: 25 in + 313 out = 338 total
Time: 10.02s

Model: mistralai/mistral-small-2603

Response:
Recursion is like a magic trick where a problem solves itself by breaking it into smaller, identical problems—like a mirror reflecting another mirror. For example, if you want to count how many steps are in a set of stairs, you can count one step and then ask how many are left, doing the same thing again. The trick stops when you reach the last step, where the answer is easy—just like how a mirror stops when it hits the wall!

Tokens: 30 in + 9

### Observations

Notice how:
- Different models give **different answers** to the same prompt
- Token counts vary (different tokenizers!)
- Response time varies (model size, provider load)
- All use the **exact same API interface** — that's the power of OpenRouter

### Try It Yourself
Change the prompt in the cell below and re-run!

In [ ]:
# YOUR TURN: Try your own prompt!
my_prompt = "Napiši mi haiku o siru. Na hrvatskom."  # <-- Change this!

response = chat([{"role": "user", "content": my_prompt}], model=MODEL_NEMOTRON)
print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

Sir leži hladan  Miris se širi tiho  
Sretan dan počinje

Tokens used: 1616


---

## Exercise 2: The Blank Slate

### Goal
Understand that LLMs have **NO memory** between API calls. Each call is completely independent. To have a "conversation," **YOU** must send the full history every time.

### Key Insight
The model is like talking to someone with amnesia — every API call is a fresh start. There is no session, no cookies, no state on the server. The `messages` array **IS** the memory.

In [ ]:
# Call 1: Tell the model your name
response1 = chat([
    {"role": "user", "content": "My name is Alice and I study Computer Science at FERIT."}
])
print("Call 1 — Model's response:")
print(response1.choices[0].message.content)

print(f"\n{'=' * 70}")

# Call 2: Ask it what your name is (SEPARATE API call!)
response2 = chat([
    {"role": "user", "content": "What is my name and what do I study?"}
])
print("\nCall 2 — Model's response:")
print(response2.choices[0].message.content)
print("\n→ The model has NO IDEA! Each call is independent. The slate is blank.")

Call 1 — Model's response:
Nice to meet you, Alice! That's great — Computer Science is such a dynamic and exciting field. FERIT (Josip Juraj Strossmayer University of Osijek) has a strong reputation in tech education in Croatia.

Are you focusing on any particular area within CS right now — like software engineering, AI, cybersecurity, or maybe something else? How are you finding your studies so far? 😊


Call 2 — Model's response:
I don't have access to personal information about you unless you've shared it with me in this specific conversation. Since this is a new session, I don't know your name or what you study.

If you tell me your name and your field of study, I'd be happy to remember it for the rest of our conversation!

→ The model has NO IDEA! Each call is independent. The slate is blank.


### The Solution: Send the Full Conversation History

To maintain context, you must include the **entire conversation** in each request:

```python
messages = [
    {"role": "user", "content": "My name is Alice..."},        # Turn 1
    {"role": "assistant", "content": "Nice to meet you..."},   # Model's reply to turn 1
    {"role": "user", "content": "What is my name?"},           # Turn 2
]
```

The model sees ALL messages and can "remember" the conversation. But it's not real memory — it's just reading the text you sent.

In [ ]:
# Now let's do it RIGHT — include the full conversation history
messages_with_history = [
    {"role": "user", "content": "My name is Alice and I study Computer Science at FERIT."},
    {"role": "assistant", "content": response1.choices[0].message.content},  # include previous response!
    {"role": "user", "content": "What is my name and what do I study?"}
]

response3 = chat(messages_with_history)
print("With full history, the model responds:")
print(response3.choices[0].message.content)
print("\n→ Now it knows! Because we sent the ENTIRE conversation.")
print(f"\nWe sent {len(messages_with_history)} messages ({response3.usage.prompt_tokens} prompt tokens)")

With full history, the model responds:
Your name is **Alice**, and you study **Computer Science at FERIT** (Josip Juraj Strossmayer University of Osijek). 😊

→ Now it knows! Because we sent the ENTIRE conversation.

We sent 3 messages (133 prompt tokens)


In [ ]:
class Conversation:
    """
    Manages a multi-turn conversation with an LLM.
    Accumulates messages so the model has full context on every call.
    """

    def __init__(self, model=MODEL_FAST, system_prompt=None):
        self.model = model
        self.messages = []
        self.total_tokens = 0
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def say(self, user_message, **kwargs):
        """Send a message and get a response. History is maintained automatically."""
        self.messages.append({"role": "user", "content": user_message})

        response = chat(self.messages, model=self.model, **kwargs)
        assistant_message = response.choices[0].message.content

        self.messages.append({"role": "assistant", "content": assistant_message})

        self.total_tokens += response.usage.total_tokens

        return assistant_message, self.total_tokens

    def get_history(self):
        """Return the full message history."""
        return self.messages.copy()

print("✓ Conversation class defined")

✓ Conversation class defined


In [ ]:
# Multi-turn conversation using our helper
conv = Conversation()

print("Turn 1:")
reply, usage = conv.say("My name is Bob. I'm learning about LLMs.")
print(reply)
print(f"  [{conv.total_tokens} tokens]")

print(f"\n{'=' * 70}\nTurn 2:")
reply, usage = conv.say("What am I learning about?")
print(reply)
print(f"  [{conv.total_tokens} tokens]")

print(f"\n{'=' * 70}\nTurn 3:")
reply, usage = conv.say("Summarize our conversation so far in one sentence.")
print(reply)
print(f"  [{conv.total_tokens} tokens]")

print(f"\n{'=' * 70}")
print(f"Messages in history: {len(conv.get_history())}")
print("→ Notice how token count GROWS with each turn — we're re-sending everything!")

Turn 1:
Hello, Bob! Welcome to the world of Large Language Models (LLMs). It's a fascinating and rapidly evolving field.

To start, here's a simple breakdown:

**What is an LLM?**
An LLM is a type of artificial intelligence (specifically, a deep learning model) trained on a massive amount of text data (books, articles, websites, etc.). It learns patterns, grammar, facts, and reasoning from this data, allowing it to generate human-like text, translate languages, write code, answer questions, and more.

**How do they work (in a nutshell)?**
1.  **Training:** They predict the next word in a sequence, over and over, on trillions of examples. This builds a complex statistical understanding of language.
2.  **Architecture:** Most modern LLMs (like GPT, Claude, Llama) use a transformer architecture, which is excellent at understanding context and relationships between words in a sentence, even over long distances.
3.  **Fine-Tuning & Alignment:** After initial training, they are often fine-tu

In [ ]:
# What happens when we remove a message from the middle?
print("=== Full history (working) ===")
full_history = [
    {"role": "user", "content": "The secret code is BLUE42. Remember it."},
    {"role": "assistant", "content": "Got it! I've noted the secret code BLUE42."},
    {"role": "user", "content": "Now tell me: what is a binary tree?"},
    {"role": "assistant", "content": "A binary tree is a data structure where each node has at most two children, called left and right."},
    {"role": "user", "content": "What was the secret code I told you earlier?"}
]
resp_full = chat(full_history)
print(resp_full.choices[0].message.content)

print(f"\n{'=' * 70}")
print("=== History with first exchange removed (broken) ===")
broken_history = [
    {"role": "user", "content": "Now tell me: what is a binary tree?"},
    {"role": "assistant", "content": "A binary tree is a data structure where each node has at most two children, called left and right."},
    {"role": "user", "content": "What was the secret code I told you earlier?"}
]
resp_broken = chat(broken_history)
print(resp_broken.choices[0].message.content)
print("\n→ Without the earlier messages, the model has no idea about the secret code!")

=== Full history (working) ===
The secret code you provided earlier is **BLUE42**.

=== History with first exchange removed (broken) ===
I don't have memory of past conversations, so I can't recall any secret code you might have mentioned earlier. Each chat session is isolated for privacy and security reasons.

If you'd like to share a code again for context, you can—but please **never share actual sensitive passwords, encryption keys, or personal secrets** in any chat. For security, it's best to keep such information private.

If you're referring to a concept (like a "secret code" in a puzzle or programming exercise), feel free to re-explain it and I’ll help!

→ Without the earlier messages, the model has no idea about the secret code!


---

## Exercise 3: System Prompts & Personas

### Goal
Learn how the **system prompt** controls the model's behavior, personality, and response style.

### Key Concept
The `system` role sets instructions **before** the conversation begins. The model treats it as its identity and operating instructions.

```python
messages = [
    {"role": "system", "content": "You are a helpful tutor..."},   # ← THIS controls behavior
    {"role": "user", "content": "Explain APIs to me"}
]
```

In [ ]:
# Same question, 4 different system prompts — watch the output change dramatically
user_question = "What is an API?"

system_prompts = {
    "No system prompt": None,
    "Patient Tutor": "You are a patient university tutor. Explain concepts simply using everyday analogies. Keep answers under 3 sentences.",
    "Software Architect": "You are a senior software architect with 20 years of experience. Give precise, technical answers. Use proper terminology. Be concise.",
    "Pirate Captain": "You are a pirate captain who explains technology using nautical metaphors. Stay in character at all times! Arrr!"
}

for name, sys_prompt in system_prompts.items():
    messages = []
    if sys_prompt:
        messages.append({"role": "system", "content": sys_prompt})
    messages.append({"role": "user", "content": user_question})

    response = chat(messages)

    print(f"\n{'=' * 70}")
    print(f"PERSONA: {name}")
    print('=' * 70)
    if sys_prompt:
        print(f"System: \"{sys_prompt[:80]}...\"")
    print(f"\n{response.choices[0].message.content}")


PERSONA: No system prompt

Excellent question! An **API (Application Programming Interface)** is one of the most important concepts in modern software, but it's often explained in a confusing way. Let's break it down.

### The Simple Analogy: A Restaurant
Imagine you're at a restaurant.
*   **You** are the **user** (or one application).
*   The **kitchen** is the **server/system** that holds all the data and performs complex operations (like cooking a meal or processing a payment).
*   The **menu and the waiter** are the **API**.

You don't walk into the kitchen, grab ingredients, and start cooking. Instead, you:
1.  **Read the menu** (the API *documentation* tells you what's available).
2.  **Place your order** with the waiter (you make an **API request**).
3.  The waiter takes your order to the kitchen, gets the meal, and brings it back (the **API processes the request** with the server).
4.  You receive your food (you get the **API response**).

The **API is the contract and the me

### Your Turn: Craft a Quiz Master System Prompt

Now create a system prompt for a **Quiz Master** — we'll use this later!

A good quiz master prompt should specify:
- The role (you are a quiz master)
- The tone (encouraging? strict?)
- Constraints (difficulty level, question style)
- We'll add the output format in Exercise 4

In [ ]:
# YOUR TURN: Write a quiz master system prompt
QUIZ_MASTER_PROMPT = """You are a Quiz Master for university students.
Your job is to generate challenging but fair quiz questions.
- Questions should test understanding, not just memorization
- Always provide 4 options (A, B, C, D)
- Include a brief explanation for the correct answer
- Be encouraging and educational in tone
"""

# Test it!
messages = [
    {"role": "system", "content": QUIZ_MASTER_PROMPT},
    {"role": "user", "content": "Generate a quiz question about Python programming."}
]
response = chat(messages)
print(response.choices[0].message.content)
print("\n→ It works! But how do we PARSE this in code? That's Exercise 4...")

**Question:**  
Consider the following list comprehension attempts in Python. Which one will raise a **SyntaxError**?

A) `[x if x > 0 else 0 for x in range(-5, 5)]`  
B) `[x for x in range(10) if x % 2 == 0 else x]`  
C) `[x**2 for x in range(5) if x % 2]`  
D) `[x*2 for x in [1,2,3] if x > 1]`  

---

**Correct Answer:** B

**Explanation:**  
Option B is invalid because in a list comprehension, the `if-else` conditional expression (ternary operator) must appear **before** the `for` clause when used as part of the output expression. The correct syntax is `[<output_expr> if <condition> else <other_expr> for <item> in <iterable>]`. Option B incorrectly places `else` after the filter `if x % 2 == 0`, which confuses the parser—it expects the `for` clause immediately after a filter `if`.  

- A is valid: ternary operator in output expression, filter is omitted.  
- C is valid: filter `if x % 2` (truthy for odd x) after `for`.  
- D is valid: simple filter after `for`.  

This tests underst

---

## Exercise 4: Structured Output & Parsing

### Goal
Learn to get LLMs to produce **machine-parseable output**. This is essential for building real applications — you need to **extract data** from LLM responses, not just display text.

### The Problem
In Exercise 3, the quiz master generated a nice-looking quiz question. But it's just free-form text. Our quiz game needs **structured data**: the question, the options, the correct answer, the explanation — all as **separate fields** we can work with in code.

### Our Approach: XML
We'll use XML tags because:
- LLMs are very good at generating well-formed XML
- Python has built-in XML parsing (`xml.etree.ElementTree`)
- Clear opening/closing tags reduce parsing errors
- It's more robust than trying to regex free-form text

In [ ]:
# The WRONG way: ask for a quiz question without specifying format
response = chat([
    {"role": "user", "content": "Generate a quiz question about computer networks with 4 options and the correct answer."}
])

raw_text = response.choices[0].message.content
print("=== Raw LLM Output ===")
print(raw_text)
print("\n" + "=" * 70)
print("THE PROBLEM: How do we extract the question, options, and correct")
print("answer from this? Every model formats it differently. Every run might")
print("be different. Parsing free-form text is fragile and unreliable!")

=== Raw LLM Output ===
**Question:**  
In the OSI model, which layer is responsible for logical addressing (such as IP addresses) and routing data across different networks?  

**Options:**  
A) Data Link Layer  
B) Network Layer  
C) Presentation Layer  
D) Session Layer  

**Correct Answer:**  
B) Network Layer

THE PROBLEM: How do we extract the question, options, and correct
answer from this? Every model formats it differently. Every run might
be different. Parsing free-form text is fragile and unreliable!


### The XML Schema

We'll ask the model to output quiz questions in this **exact** format:

```xml
<quiz>
  <question>What does HTTP stand for?</question>
  <options>
    <option id="A">HyperText Transfer Protocol</option>
    <option id="B">High-Level Text Processing</option>
    <option id="C">Hyperlink and Text Transfer Process</option>
    <option id="D">Home Tool Transfer Protocol</option>
  </options>
  <correct>A</correct>
  <explanation>HTTP stands for HyperText Transfer Protocol, the foundation of web communication.</explanation>
</quiz>
```

Now Python can parse this reliably every time.

In [ ]:
# The XML-structured system prompt — notice how PRECISE it is
QUIZ_XML_SYSTEM_PROMPT = """You are a Quiz Master that generates quiz questions in XML format.

You MUST respond with ONLY valid XML in this exact format — no other text before or after:

<quiz>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <explanation>Brief explanation of why the correct answer is correct.</explanation>
</quiz>

Rules:
- Always exactly 4 options labeled A, B, C, D
- The <correct> tag must contain exactly one letter: A, B, C, or D
- The <explanation> must explain WHY the answer is correct
- Questions should be challenging but fair for university students
- Wrong options should be plausible, not obviously silly
- Do NOT include any text outside the <quiz> tags
"""

# Generate a structured quiz question
response = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": "Generate a quiz question about operating systems."}
])

xml_text = response.choices[0].message.content
print("=== Structured XML Output ===")
print(xml_text)

=== Structured XML Output ===
<quiz>
  <question>In the context of process synchronization, what is the primary conceptual difference between a mutex and a binary semaphore?</question>
  <options>
    <option id="A">A mutex can only be used by the thread that locked it, while a binary semaphore can be signaled by any thread.</option>
    <option id="B">A mutex is used for signaling between threads, while a binary semaphore is used for mutual exclusion.</option>
    <option id="C">A mutex is a kernel-level object, while a binary semaphore is always a user-level construct.</option>
    <option id="D">A mutex has a concept of ownership, whereas a binary semaphore does not.</option>
  </options>
  <correct>D</correct>
  <explanation>The key distinction is ownership. A mutex is owned by the thread that locks it; only that thread can unlock it, which enforces strict mutual exclusion and prevents priority inversion. A binary semaphore (value 0 or 1) is a signaling mechanism without ownership—

In [ ]:
def parse_quiz_xml(xml_string):
    """
    Parse a quiz question from XML format into a Python dict.

    Args:
        xml_string: XML string containing a quiz question

    Returns:
        Dict with keys: question, options (dict A-D), correct, explanation
        Returns None if parsing fails
    """
    try:
        # Clean up: strip any text before <quiz> and after </quiz>
        start = xml_string.find("<quiz>")
        end = xml_string.find("</quiz>")
        if start == -1 or end == -1:
            print("ERROR: Could not find <quiz>...</quiz> tags")
            return None

        clean_xml = xml_string[start:end + len("</quiz>")]
        root = ET.fromstring(clean_xml)

        question = root.find("question").text.strip()

        options = {}
        for opt in root.find("options").findall("option"):
            options[opt.get("id")] = opt.text.strip()

        correct = root.find("correct").text.strip()
        explanation = root.find("explanation").text.strip()

        # Validate
        if correct not in ["A", "B", "C", "D"]:
            print(f"WARNING: Invalid correct answer '{correct}', expected A-D")
            return None
        if len(options) != 4:
            print(f"WARNING: Expected 4 options, got {len(options)}")
            return None

        return {
            "question": question,
            "options": options,
            "correct": correct,
            "explanation": explanation
        }
    except ET.ParseError as e:
        print(f"XML Parse Error: {e}")
        print(f"Raw text was:\n{xml_string[:300]}...")
        return None
    except AttributeError as e:
        print(f"Missing XML element: {e}")
        return None

print("✓ parse_quiz_xml() function defined")

✓ parse_quiz_xml() function defined


In [ ]:
# Parse the XML we generated and display it as structured data
quiz = parse_quiz_xml(xml_text)

if quiz:
    print("=== Parsed Quiz Question ===")
    print(f"\nQ: {quiz['question']}\n")
    for letter, text in quiz['options'].items():
        print(f"  {letter}) {text}")
    print(f"\nCorrect answer: {quiz['correct']}")
    print(f"Explanation: {quiz['explanation']}")
    print("\n→ Now we have clean, structured data we can use in code!")
else:
    print("Failed to parse! Let's see what went wrong...")
    print(xml_text)

=== Parsed Quiz Question ===

Q: In the context of process synchronization, what is the primary conceptual difference between a mutex and a binary semaphore?

  A) A mutex can only be used by the thread that locked it, while a binary semaphore can be signaled by any thread.
  B) A mutex is used for signaling between threads, while a binary semaphore is used for mutual exclusion.
  C) A mutex is a kernel-level object, while a binary semaphore is always a user-level construct.
  D) A mutex has a concept of ownership, whereas a binary semaphore does not.

Correct answer: D
Explanation: The key distinction is ownership. A mutex is owned by the thread that locks it; only that thread can unlock it, which enforces strict mutual exclusion and prevents priority inversion. A binary semaphore (value 0 or 1) is a signaling mechanism without ownership—any thread can wait (decrement) or signal (increment) it, making it suitable for event notification between unrelated threads.

→ Now we have clean, 

In [ ]:
def play_quiz(quiz_data):
    """
    Play a single quiz question interactively.

    Args:
        quiz_data: Parsed quiz dict from parse_quiz_xml()

    Returns:
        True if answered correctly, False otherwise
    """
    print(f"\n{'=' * 60}")
    print(f"  {quiz_data['question']}")
    print('=' * 60)

    for letter, text in quiz_data['options'].items():
        print(f"  {letter}) {text}")

    answer = input("\nYour answer (A/B/C/D): ").strip().upper()

    if answer == quiz_data['correct']:
        print("\n✓ CORRECT!")
    else:
        print(f"\n✗ Incorrect. The correct answer is {quiz_data['correct']}.")

    print(f"\nExplanation: {quiz_data['explanation']}")
    return answer == quiz_data['correct']

print("✓ play_quiz() function defined")

✓ play_quiz() function defined


In [ ]:
# What happens with a VAGUE prompt? Let's see it break.
print("=== Vague prompt — what could go wrong? ===")
response_vague = chat([
    {"role": "system", "content": "Generate quiz questions in XML."},  # TOO VAGUE!
    {"role": "user", "content": "Make a quiz."}
])

print(response_vague.choices[0].message.content[:500])
print("\n" + "=" * 70)
print("\nAttempting to parse...")
result = parse_quiz_xml(response_vague.choices[0].message.content)
if result is None:
    print("\n→ FAILED! A vague prompt leads to unpredictable format.")
    print("→ This is why the PRECISE system prompt with the EXACT schema matters.")
    print("→ Prompt engineering isn't about being fancy — it's about RELIABILITY.")
else:
    print("\n→ It worked this time, but try running it again — vague prompts are inconsistent.")

=== Vague prompt — what could go wrong? ===
```xml
<?xml version="1.0" encoding="UTF-8"?>
<quiz>
  <question id="1" type="multiple_choice">
    <text>What is the chemical symbol for water?</text>
    <option correct="true">H₂O</option>
    <option>CO₂</option>
    <option>O₂</option>
    <option>H₂</option>
  </question>
  <question id="2" type="true_false">
    <text>Neil Armstrong was the first human to walk on the moon.</text>
    <option correct="true">True</option>
    <option>False</option>
  </question>
  <question id="3" type="mul


Attempting to parse...
Missing XML element: 'NoneType' object has no attribute 'findall'

→ FAILED! A vague prompt leads to unpredictable format.
→ This is why the PRECISE system prompt with the EXACT schema matters.
→ Prompt engineering isn't about being fancy — it's about RELIABILITY.


In [ ]:
# Full flow: generate → parse → play!
print("Let's play a quiz question!\n")

response = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": "Generate a quiz question about data structures."}
])

quiz = parse_quiz_xml(response.choices[0].message.content)
if quiz:
    play_quiz(quiz)

Let's play a quiz question!


  Which data structure provides O(1) time complexity for insertion and deletion at both the beginning and the end?
  A) Array
  B) Singly Linked List
  C) Doubly Linked List
  D) Circular Queue

Your answer (A/B/C/D): A

✗ Incorrect. The correct answer is C.

Explanation: A doubly linked list maintains pointers to both the next and previous nodes, allowing constant-time adjustments for insertion or deletion at the head or tail. Arrays require O(n) time due to element shifting, singly linked lists lack backward pointers for efficient tail operations (deletion at tail is O(n)), and circular queues are designed for FIFO operations with enqueue and dequeue at opposite ends, not arbitrary end access.


---

## Exercise 5: Thinking Models & Prompt Engineering

### Goal
Compare standard models with **thinking/reasoning models** that show their chain of thought. Then see how **prompt engineering** dramatically affects output quality.

### What Are Thinking Models?
Some models (like `openai/gpt-oss-120b`) have a built-in reasoning step:
1. The model first **THINKS** through the problem (chain-of-thought)
2. Then it produces the final answer
3. You can see the **reasoning tokens** — the model's "inner monologue"

This produces better answers for complex tasks, but uses **significantly more tokens** (and costs more).

> **Note:** If `gpt-oss-120b` is slow or unavailable, you can skip the thinking model cell — the prompt engineering exercises below work with any model.

In [ ]:
# A task that benefits from reasoning
task = """Generate a quiz question about the difference between TCP and UDP
that tests DEEP understanding, not just definitions.
Present a realistic scenario where choosing the right protocol matters."""

# Standard model first
print("=== Standard Model (Step Flash) ===")
start = time.time()
response_standard = chat(
    [{"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
     {"role": "user", "content": task}],
    model=MODEL_FAST
)
time_standard = time.time() - start

quiz_standard = parse_quiz_xml(response_standard.choices[0].message.content)
if quiz_standard:
    print(f"Q: {quiz_standard['question']}")
    for k, v in quiz_standard['options'].items():
        print(f"  {k}) {v}")
print(f"\nTokens: {response_standard.usage.total_tokens} | Time: {time_standard:.1f}s")

=== Standard Model (Step Flash) ===
Q: In a real-time multiplayer online game, why is UDP typically chosen for transmitting player positions and actions rather than TCP?
  A) UDP ensures that all player movements are delivered in the correct order.
  B) UDP minimizes latency, allowing for smoother gameplay despite occasional packet loss.
  C) TCP's retransmission mechanism prevents lag during network congestion.
  D) UDP provides built-in encryption for secure game data.

Tokens: 2605 | Time: 30.8s


In [ ]:
# Now the THINKING model — watch the reasoning tokens!
print("=== Thinking Model (GPT-OSS-120B) ===")
print("Generating... (this may take longer)\n")

start = time.time()
response_thinking = chat(
    [{"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
     {"role": "user", "content": task}],
    model=MODEL_THINKING,
    extra_body={
        "reasoning": {
            "effort": "high"
        }
    }
)
time_thinking = time.time() - start

# Access the reasoning (chain-of-thought) — the model's inner monologue
reasoning = getattr(response_thinking.choices[0].message, 'reasoning', None)

if reasoning:
    print("--- REASONING (model's inner thoughts) ---")
    # Show first 1500 chars of reasoning
    print(reasoning[:1500])
    if len(reasoning) > 1500:
        print(f"\n... [{len(reasoning)} total characters of reasoning]")
    print("\n--- END REASONING ---\n")
else:
    print("(No reasoning field returned — model may not support it via this provider)\n")

print("--- FINAL ANSWER ---")
quiz_thinking = parse_quiz_xml(response_thinking.choices[0].message.content)
if quiz_thinking:
    print(f"Q: {quiz_thinking['question']}")
    for k, v in quiz_thinking['options'].items():
        print(f"  {k}) {v}")
print(f"\nTokens: {response_thinking.usage.total_tokens} | Time: {time_thinking:.1f}s")

=== Thinking Model (GPT-OSS-120B) ===
Generating... (this may take longer)

--- REASONING (model's inner thoughts) ---
The user wants a quiz question about the difference between TCP and UDP that tests deep understanding, not just definitions, with a realistic scenario where choosing the right protocol matters.

We need to output only valid XML with the exact structure:

<quiz>
  <question>...</question>
  <options>
    <option id="A">...</option>
    <option id="B">...</option>
    <option id="C">...</option>
    <option id="D">...</option>
  </options>
  <correct>A</correct>
  <explanation>...</explanation>
</quiz>

No extra text. Must adhere exactly. The question must be about the difference between TCP and UDP, deep understanding, realistic scenario where choosing the right protocol matters.

We need to ensure it's challenging for university students, plausible wrong options.

We need to decide on the correct answer (A, B, C, or D). Let's craft a scenario.

Possible scenario: Real-

In [ ]:
# Token usage comparison — side by side
print("=== Token Usage Comparison ===")
print(f"{'':30} {'Standard':>12} {'Thinking':>12}")
print("-" * 56)
print(f"{'Prompt tokens':30} {response_standard.usage.prompt_tokens:>12} {response_thinking.usage.prompt_tokens:>12}")
print(f"{'Completion tokens':30} {response_standard.usage.completion_tokens:>12} {response_thinking.usage.completion_tokens:>12}")
print(f"{'Total tokens':30} {response_standard.usage.total_tokens:>12} {response_thinking.usage.total_tokens:>12}")
print(f"{'Time (seconds)':30} {time_standard:>12.1f} {time_thinking:>12.1f}")

ratio = response_thinking.usage.total_tokens / max(response_standard.usage.total_tokens, 1)
print(f"\n→ Thinking model used {ratio:.1f}x more tokens!")
print("→ Reasoning is powerful but expensive — use it for tasks that need it.")
print("→ The thinking tokens are the model's 'chain of thought' — it reasons before answering.")

=== Token Usage Comparison ===
                                   Standard     Thinking
--------------------------------------------------------
Prompt tokens                           266          309
Completion tokens                      2339         2500
Total tokens                           2605         2809
Time (seconds)                         30.8         30.8

→ Thinking model used 1.1x more tokens!
→ Reasoning is powerful but expensive — use it for tasks that need it.
→ The thinking tokens are the model's 'chain of thought' — it reasons before answering.


### Prompt Engineering: Lazy vs. Engineered

The quality of your prompt **dramatically** affects the quality of the output. Let's prove it.

In [ ]:
# ROUND 1: LAZY prompt
lazy_prompt = "Make a quiz question about sorting algorithms."

# ROUND 2: ENGINEERED prompt
engineered_prompt = """Generate a quiz question about sorting algorithms that:
- Tests understanding of TIME COMPLEXITY trade-offs (not just "what is quicksort")
- Presents a realistic scenario (e.g., choosing the right algorithm for a specific dataset)
- Has plausible distractors that represent common misconceptions
- The wrong answers should be algorithms that COULD seem correct but aren't optimal for the scenario
- Difficulty: university-level data structures course"""

print("=== LAZY PROMPT ===")
print(f"Prompt: \"{lazy_prompt}\"\n")
resp_lazy = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": lazy_prompt}
])
quiz_lazy = parse_quiz_xml(resp_lazy.choices[0].message.content)
if quiz_lazy:
    print(f"Q: {quiz_lazy['question']}")
    for k, v in quiz_lazy['options'].items():
        print(f"  {k}) {v}")

print(f"\n{'=' * 70}")
print("\n=== ENGINEERED PROMPT ===")
print(f"Prompt: \"{engineered_prompt[:80]}...\"\n")
resp_eng = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": engineered_prompt}
])
quiz_eng = parse_quiz_xml(resp_eng.choices[0].message.content)
if quiz_eng:
    print(f"Q: {quiz_eng['question']}")
    for k, v in quiz_eng['options'].items():
        print(f"  {k}) {v}")

print("\n→ Compare: which question tests deeper understanding?")
print("→ Which has better distractors? More realistic scenario?")
print("→ The engineered prompt didn't cost more tokens — just more thought.")

=== LAZY PROMPT ===
Prompt: "Make a quiz question about sorting algorithms."

Q: Which of the following sorting algorithms is NOT stable in its typical implementation?
  A) Merge sort
  B) Insertion sort
  C) Quick sort
  D) Bubble sort


=== ENGINEERED PROMPT ===
Prompt: "Generate a quiz question about sorting algorithms that:
- Tests understanding of..."

Q: You are tasked with sorting 500,000 records where each record is a 10-digit phone number (as a string of digits). The data is randomly ordered, but you know that all phone numbers share the same 3-digit area code (e.g., "415-***-****"). Which algorithm provides the best worst-case time complexity for this specific dataset?
  A) Quicksort with a median-of-three pivot
  B) Merge sort
  C) Radix sort (LSD, base 10)
  D) Heapsort

→ Compare: which question tests deeper understanding?
→ Which has better distractors? More realistic scenario?
→ The engineered prompt didn't cost more tokens — just more thought.


In [ ]:
# Same engineered prompt across multiple models — compare quality
print("=== Multi-Model Comparison (same engineered prompt) ===")

for model_name in [MODEL_FAST, MODEL_MISTRAL, MODEL_NEMOTRON, MODEL_MINISTRAL]:
    resp = chat([
        {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
        {"role": "user", "content": engineered_prompt}
    ], model=model_name)

    quiz = parse_quiz_xml(resp.choices[0].message.content)
    print(f"\n{'=' * 70}")
    print(f"Model: {model_name}")
    print('=' * 70)
    if quiz:
        print(f"Q: {quiz['question']}")
        for k, v in quiz['options'].items():
            print(f"  {k}) {v}")
        print(f"Answer: {quiz['correct']}")
    else:
        print("Failed to parse XML output")
    print(f"Tokens: {resp.usage.total_tokens}")

=== Multi-Model Comparison (same engineered prompt) ===

Model: stepfun/step-3.5-flash:free
Q: You are given an array of 1,000,000 integers that is nearly sorted (i.e., the number of inversions is O(n)). You must sort this array completely, and your primary concern is minimizing the number of comparisons and swaps. Which algorithm is the most optimal choice?
  A) Insertion Sort
  B) Quicksort with a random pivot
  C) Merge Sort
  D) Heapsort
Answer: A
Tokens: 9820

Model: mistralai/mistral-small-2603
Q: You are designing a real-time analytics dashboard that must update sorted rankings of a dataset of 1,000,000 user scores every 5 seconds. The scores are updated frequently, but the rankings only need to be accurate to within the top 100 results. Which sorting algorithm would provide the best time complexity trade-off for this scenario?
  A) Use a min-heap to maintain the top 100 scores, allowing O(n log k) insertion where k=100.
  B) Apply quicksort to the entire dataset every 5 seconds

---

## Exercise 6: Study Buddy — Bring Your Own Knowledge (Capstone)

### Goal
Build a complete quiz game from **YOUR OWN study notes**! This ties together everything:

| Concept | From Exercise | Used Here |
|---------|--------------|----------|
| API calls & tokens | Exercise 1 | Every generation call |
| Conversation history | Exercise 2 | No repeated questions |
| System prompts | Exercise 3 | Quiz master + knowledge base |
| XML parsing | Exercise 4 | Parse every question |
| Prompt engineering | Exercise 5 | Quality of generated questions |

### How It Works
1. You upload (or paste) your study notes in markdown
2. The **entire text** is injected into the system prompt as context
3. The model generates quiz questions **from your material only**
4. Conversation history prevents repeated questions
5. You play the quiz interactively!

In [ ]:
# OPTION 1: Upload a markdown/text file with your study notes
# (skip this cell and use the next one if upload doesn't work)

study_notes = None

try:
    from google.colab import files
    print("Upload your study notes (.md or .txt file):")
    print("(Or skip this cell and use the next one to paste text directly)\n")
    uploaded = files.upload()

    if uploaded:
        filename = list(uploaded.keys())[0]
        study_notes = uploaded[filename].decode('utf-8')
        print(f"\n✓ Loaded '{filename}' ({len(study_notes)} characters, ~{len(study_notes)//4} tokens)")
except Exception as e:
    print(f"Upload not available: {e}")
    print("Use the next cell to paste your notes directly.")

In [ ]:
# OPTION 2: Paste your notes here (or use this sample if you don't have notes handy)

if study_notes is None:
    study_notes = """
# Python Data Structures

## Lists
Lists are ordered, mutable sequences. They allow duplicate elements.
- Access by index: O(1)
- Search: O(n)
- Append: O(1) amortized
- Insert at position: O(n)
Lists are implemented as dynamic arrays internally.

## Tuples
Tuples are ordered, immutable sequences. Once created, they cannot be modified.
- Faster than lists for iteration
- Can be used as dictionary keys (because they are hashable)
- Memory efficient — Python can cache and reuse small tuples

## Dictionaries
Dictionaries store key-value pairs. Since Python 3.7, they maintain insertion order.
- Implemented as hash tables
- Average lookup: O(1)
- Worst case lookup: O(n) — hash collisions
- Keys must be hashable (immutable types like str, int, tuple)
- dict comprehensions: {k: v for k, v in items}

## Sets
Sets are unordered collections of unique elements.
- Implemented as hash tables (like dict, but keys only)
- Membership test: O(1) average
- Support set operations: union (|), intersection (&), difference (-), symmetric difference (^)
- No duplicate elements — adding a duplicate is silently ignored
- frozenset is the immutable version (can be used as dict keys)

## Collections module
- defaultdict: dict subclass with a factory function for missing keys
- Counter: dict subclass for counting hashable objects
- deque: double-ended queue with O(1) append/pop from both ends
- namedtuple: factory function for creating tuple subclasses with named fields
- OrderedDict: dict that remembers insertion order (less useful since Python 3.7)

## When to Use What
- Need ordered data with duplicates? → List
- Need immutable sequence (dict key, set element)? → Tuple
- Need fast key-value lookup? → Dictionary
- Need unique elements and set operations? → Set
- Need fast append/pop from both ends? → deque
- Need to count occurrences? → Counter
"""
    print("Using sample notes: 'Python Data Structures'")

print(f"\nNotes: {len(study_notes)} characters (~{len(study_notes)//4} tokens)")
print(f"\n=== Preview ===")
print(study_notes[:500])
if len(study_notes) > 500:
    print(f"\n... [{len(study_notes)} total characters]")

In [ ]:
# Build the Study Buddy system prompt — note how we inject the FULL notes
STUDY_BUDDY_PROMPT = f"""You are a Study Buddy Quiz Master. Your job is to generate quiz questions
based ONLY on the study material provided below.

STUDY MATERIAL:
---
{study_notes}
---

RULES:
1. Generate questions ONLY from the material above — do NOT use outside knowledge
2. Test understanding and application, not just memorization
3. Make wrong options plausible (common misconceptions about these topics)
4. Vary the difficulty across questions
5. Do NOT repeat a question that has already been asked in this conversation
6. Cover different sections of the material across questions

Respond with ONLY valid XML in this exact format:

<quiz>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <explanation>Brief explanation referencing the study material.</explanation>
</quiz>
"""

# How big is this prompt?
prompt_tokens_est = len(STUDY_BUDDY_PROMPT) // 4
notes_tokens_est = len(study_notes) // 4
print(f"System prompt size: ~{prompt_tokens_est} tokens")
print(f"  of which study notes: ~{notes_tokens_est} tokens")
print(f"\n→ This system prompt is sent with EVERY request.")
print(f"→ Larger notes = more tokens per call = higher cost.")
print(f"→ This is exactly the problem that RAG solves (next session)!")

In [ ]:
# Generate and play your first Study Buddy question!
quiz_session = Conversation(model=MODEL_FAST, system_prompt=STUDY_BUDDY_PROMPT)

print("Generating your first quiz question from your notes...\n")
xml_response, usage = quiz_session.say("Generate a quiz question.")

quiz = parse_quiz_xml(xml_response)
if quiz:
    play_quiz(quiz)
    print(f"\n[Tokens used: {usage.total_tokens}]")
else:
    print("Failed to parse. Raw response:")
    print(xml_response)

In [ ]:
# Play 3 more rounds — the model won't repeat questions because
# the full conversation history is sent each time (Exercise 2!)
score = 0
rounds = 3

for round_num in range(1, rounds + 1):
    print(f"\n{'#' * 60}")
    print(f"   ROUND {round_num} of {rounds}")
    print('#' * 60)

    xml_response, usage = quiz_session.say(
        "Generate another quiz question on a DIFFERENT topic from the previous ones."
    )
    quiz = parse_quiz_xml(xml_response)

    if quiz:
        correct = play_quiz(quiz)
        if correct:
            score += 1
        print(f"\n[Tokens this round: {usage.total_tokens}]")
    else:
        print("Failed to generate a valid question. Skipping...")

print(f"\n{'=' * 60}")
print(f"   FINAL SCORE: {score}/{rounds}")
print(f"   Messages in conversation: {len(quiz_session.get_history())}")
print('=' * 60)

### Token Scaling

Notice how each round uses **more tokens** than the last. That's because the **full conversation history** grows with each exchange.

This is the fundamental trade-off:
- **More history** = better context, no repeated questions
- **More history** = more tokens per call, higher cost, eventually hits context window limit

In production, you'd address this with:
- **Sliding window** — only keep the last N messages
- **Summarization** — summarize old turns into a compact form
- **RAG** — retrieve only relevant context instead of sending everything (next session!)

In [ ]:
# Visualize how tokens scale with conversation length
print("=== Token Scaling Across the Conversation ===")
print("(estimated from character count)\n")

history = quiz_session.get_history()
running_chars = 0
for i, msg in enumerate(history):
    running_chars += len(msg["content"])
    estimated_tokens = running_chars // 4
    bar = "█" * (estimated_tokens // 100)
    role = msg["role"]
    print(f"  Msg {i+1:2d} ({role:10s}): ~{estimated_tokens:5d} tokens  {bar}")

print(f"\n→ The system prompt ({len(STUDY_BUDDY_PROMPT)//4}+ tokens) is re-sent every single time.")
print("→ By round 3, we're sending thousands of tokens just in conversation history.")

In [ ]:
# Try with a different model — same notes, different quiz master
print(f"=== Same notes, different model: {MODEL_MISTRAL} ===")

quiz_session_ds = Conversation(model=MODEL_MISTRAL, system_prompt=STUDY_BUDDY_PROMPT)

xml_response, usage = quiz_session_ds.say("Generate a challenging quiz question.")
quiz = parse_quiz_xml(xml_response)
if quiz:
    play_quiz(quiz)
    print(f"\n[Tokens: {usage.total_tokens}]")

---

## Key Takeaways

### What We Learned

1. **API Basics (Exercise 1)** — LLMs are accessed via the Chat Completions API. Every call has: messages in, response out, token usage. Different models have different strengths, speeds, and costs.

2. **Statelessness (Exercise 2)** — LLMs have NO memory between calls. YOU manage conversation history by sending all messages each time. More history = more tokens per call.

3. **System Prompts (Exercise 3)** — The system prompt is your primary control mechanism. Same question + different system prompt = dramatically different output.

4. **Structured Output (Exercise 4)** — XML schemas give you reliable, parseable output. A precise schema in the prompt is essential. Always validate and handle parsing failures.

5. **Thinking Models (Exercise 5)** — Reasoning models show chain-of-thought and produce better answers for complex tasks, but burn significantly more tokens. Prompt engineering matters as much as model selection.

6. **Knowledge in Context (Exercise 6)** — You can inject documents into the prompt as context. Token costs scale with document size + conversation length. This works but doesn't scale.

### What's Next

In the next session, we'll tackle the scaling problem from Exercise 6 with **Retrieval-Augmented Generation (RAG)**:
- Instead of sending ALL your notes in the prompt, we'll use **embeddings** to convert text into vectors
- A **vector database** will store and search these embeddings
- Only the **relevant sections** get retrieved and sent to the model
- This is how production AI systems work at scale — ChatGPT, Perplexity, and enterprise AI assistants all use RAG

## Bonus Challenges (Optional)

1. **Multi-topic quiz** — Upload notes from multiple subjects, ask the model to generate mixed quizzes
2. **Difficulty levels** — Modify the system prompt to accept a difficulty parameter (easy/medium/hard) and test whether the model actually follows it
3. **Model tournament** — Run the same 10 questions through all 3 models, compare which produces the best questions
4. **Prompt optimization** — Experiment with adding tags like `<difficulty>`, `<topic>`, `<hint>` to the XML schema
5. **Token budget** — Build a version that summarizes old conversation turns to stay under a token budget

In [ ]:
# Your experiments here!
